# S06 — Solutions: SciPy, Pandas & Serialisation

**Python for Applied Physics** | Master in Applied Physics  
⚠️ *Instructor reference — do not distribute before the exercise session.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import h5py, json, tempfile, os
from scipy import integrate, optimize, interpolate, signal
from scipy import constants

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

def fwhm(x, y):
    yn = y / y.max(); above = yn >= 0.5
    return x[above].max() - x[above].min() if above.any() else np.nan

---
## Exercise 1 — Beam power via `quad`

In [ ]:
w0 = 350e-6
I0 = 1e6

def integrand(r):
    return 2 * np.pi * r * I0 * np.exp(-2 * r**2 / w0**2)

P_quad, P_err = integrate.quad(integrand, 0, np.inf)
P_analytic    = I0 * np.pi * w0**2 / 2

results = {}
for N_pts in [50, 200, 1000]:
    r_grid = np.linspace(0, 5*w0, N_pts)
    I_grid = I0 * np.exp(-2 * r_grid**2 / w0**2)
    P_trapz = 2 * np.pi * np.trapz(I_grid * r_grid, r_grid)
    rel_err  = abs(P_trapz - P_analytic) / P_analytic
    results[N_pts] = (P_trapz, rel_err)

assert np.isclose(P_quad, P_analytic, rtol=1e-6)

print(f"quad     : {P_quad:.8f} W  (err={P_err:.2e})")
print(f"analytic : {P_analytic:.8f} W")
print()
for N_pts, (P_t, rel_e) in results.items():
    flag = '✓' if rel_e < 1e-4 else ' '
    print(f"trapz N={N_pts:5d}: {P_t:.6f} W  rel_err={rel_e:.2e} {flag}")
print("All assertions passed ✓")

---
## Exercise 2 — Gaussian fit to a noisy beam profile

In [ ]:
rng     = np.random.default_rng(123)
w_true  = 280e-6
I_true  = 2.5e6
r0_true = 15e-6

r_data  = np.linspace(-1e-3, 1e-3, 100)
I_data  = I_true * np.exp(-2*(r_data - r0_true)**2 / w_true**2)
I_data += rng.normal(0, 0.04 * I_true, size=len(r_data))

def gaussian_model(r, I_peak, w, r0):
    return I_peak * np.exp(-2*(r - r0)**2 / w**2)

p0       = [I_data.max(), 300e-6, 0.0]
popt, pcov = optimize.curve_fit(gaussian_model, r_data, I_data, p0=p0)
perr       = np.sqrt(np.diag(pcov))
I_fit, w_fit, r0_fit = popt
σ_I, σ_w, σ_r0       = perr

r_dense  = np.linspace(r_data[0], r_data[-1], 512)
I_fitted = gaussian_model(r_dense, *popt)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(r_data*1e3,  I_data/1e6,   'o', ms=3, alpha=0.6, color='C0', label='Data')
ax.plot(r_dense*1e3, I_fitted/1e6, lw=2, color='C1',
        label=f'Fit  w={w_fit*1e6:.0f}±{σ_w*1e6:.0f} µm')
ax.set_xlabel('r (mm)'); ax.set_ylabel('I (MW/m²)')
ax.set_title('Gaussian beam fit'); ax.legend()
fig.tight_layout(); plt.show()

assert np.abs(w_fit  - w_true)  < 2*σ_w
assert np.abs(r0_fit - r0_true) < 2*σ_r0
assert np.abs(I_fit  - I_true)  < 2*σ_I

print(f"I_peak = {I_fit/1e6:.3f} ± {σ_I/1e6:.3f} MW/m²  (true: {I_true/1e6:.3f})")
print(f"w      = {w_fit*1e6:.1f} ± {σ_w*1e6:.1f} µm      (true: {w_true*1e6:.0f})")
print(f"r0     = {r0_fit*1e6:.1f} ± {σ_r0*1e6:.1f} µm    (true: {r0_true*1e6:.0f})")
print("All assertions passed ✓")

---
## Exercise 3 — Upsample a coarse camera image

In [ ]:
rng  = np.random.default_rng(55)
N_c  = 20; N_f = 200; w_px = 3

x_c = np.linspace(-10, 10, N_c)
y_c = np.linspace(-10, 10, N_c)
Xc, Yc = np.meshgrid(x_c, y_c, indexing='ij')
I_true_c = np.exp(-2*(Xc**2+Yc**2)/w_px**2)
I_coarse  = rng.poisson(I_true_c * 1000) / 1000

x_f = np.linspace(-10, 10, N_f)
y_f = np.linspace(-10, 10, N_f)
Xf, Yf   = np.meshgrid(x_f, y_f, indexing='ij')
I_true_f  = np.exp(-2*(Xf**2+Yf**2)/w_px**2)
pts_f     = np.column_stack([Xf.ravel(), Yf.ravel()])

results_interp = {}
for method in ['linear', 'cubic']:
    rgi  = interpolate.RegularGridInterpolator((x_c, y_c), I_coarse, method=method,
                                               bounds_error=False, fill_value=0.0)
    I_up = rgi(pts_f).reshape(N_f, N_f)
    rms  = np.sqrt(np.mean((I_up - I_true_f)**2))
    results_interp[method] = (I_up, rms)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
titles = ['Coarse (20×20)', 'Linear (200×200)', 'Cubic (200×200)']
images = [(I_coarse.T, x_c, y_c),
          (results_interp['linear'][0].T, x_f, y_f),
          (results_interp['cubic'][0].T,  x_f, y_f)]
for ax, title, (img, xax, yax) in zip(axes, titles, images):
    ax.imshow(img, origin='lower',
              extent=[xax[0], xax[-1], yax[0], yax[-1]],
              cmap='inferno', aspect='equal', vmin=0, vmax=1)
    ax.set_title(title); ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
fig.tight_layout(); plt.show()

for method, (_, rms) in results_interp.items():
    print(f"{method:8s}: RMS error = {rms:.5f}")

assert I_coarse.shape == (N_c, N_c)
assert results_interp['cubic'][1] <= results_interp['linear'][1]
print("All assertions passed ✓")

**Which method is better and why?**  
Cubic interpolation is more accurate for smooth data like a Gaussian beam because it uses higher-order polynomial fitting that can follow smooth curvature. Linear interpolation produces facets (piecewise-flat patches), introducing visible artefacts when upsampling by a large factor.

---
## Exercise 4 — Multi-pulse detection + spectrogram

In [ ]:
rng     = np.random.default_rng(77)
N_tr    = 16384; dt_tr = 2e-15
t_tr    = np.arange(N_tr) * dt_tr
τ_pulse = 300e-15; GDD_chirp = 5000e-30

t_positions = rng.uniform(5e-12, 25e-12, size=4)
amplitudes  = rng.uniform(0.5, 1.0, size=4)

I_trace = np.zeros(N_tr)
for t0, A in zip(t_positions, amplitudes):
    I_trace += A * np.exp(-((t_tr - t0)/τ_pulse)**2)
noise_std = 10**(-20/20)
I_trace  += rng.normal(0, noise_std, size=N_tr)

# Chirp the strongest pulse
idx_s    = np.argmax(amplitudes)
t0_chirp = t_positions[idx_s]
A_chirp  = amplitudes[idx_s]

E_single    = A_chirp * np.exp(-((t_tr - t0_chirp)/τ_pulse)**2)
freq_tr     = np.fft.fftfreq(N_tr, d=dt_tr)
ω_tr        = 2 * np.pi * freq_tr
E_f_single  = np.fft.fft(E_single)
E_f_chirped = E_f_single * np.exp(0.5j * GDD_chirp * ω_tr**2)
E_chirped   = np.fft.ifft(E_f_chirped)

I_trace -= E_single**2
I_trace += np.abs(E_chirped)**2

min_dist   = int(2e-12 / dt_tr)
peaks, _   = signal.find_peaks(I_trace, height=0.2*amplitudes.max(),
                                distance=min_dist, prominence=0.1)

fs_tr = 1 / dt_tr
f_sp, t_sp, Sxx = signal.spectrogram(I_trace, fs=fs_tr,
                                      nperseg=512, noverlap=480,
                                      scaling='spectrum')

assert len(peaks) == 4, f"Expected 4, got {len(peaks)}"
assert Sxx.ndim == 2

print(f"Detected {len(peaks)} peaks:")
for pk in peaks:
    print(f"  t = {t_tr[pk]*1e12:.2f} ps,  I = {I_trace[pk]:.3f}")

freq_mask = f_sp < 5e12
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(t_tr*1e12, I_trace, lw=0.8, color='C0')
axes[0].plot(t_tr[peaks]*1e12, I_trace[peaks], 'v', ms=10, color='C1')
axes[0].set_xlabel('Time (ps)'); axes[0].set_ylabel('Intensity')
axes[0].set_title('Multi-pulse trace')

axes[1].pcolormesh(t_sp*1e12, f_sp[freq_mask]/1e12,
                    Sxx[freq_mask,:], cmap='inferno', shading='auto')
axes[1].set_xlabel('Time (ps)'); axes[1].set_ylabel('Frequency (THz)')
axes[1].set_title('Spectrogram')
fig.tight_layout(); plt.show()
print("All assertions passed ✓")

---
## Exercise 5 — Laser shot log analysis with Pandas

In [ ]:
rng = np.random.default_rng(42); N = 120
channels     = ['signal_1310nm', 'idler_1900nm', 'SH_655nm']
true_energy  = {'signal_1310nm': 30.0, 'idler_1900nm': 45.0, 'SH_655nm': 12.0}
true_dur     = {'signal_1310nm': 80.0, 'idler_1900nm': 120.0, 'SH_655nm': 60.0}

ch_col  = rng.choice(channels, size=N)
E_col   = np.array([rng.normal(true_energy[c], true_energy[c]*0.03) for c in ch_col])
dur_col = np.array([rng.normal(true_dur[c],   true_dur[c]*0.05)   for c in ch_col])
outlier_idx = rng.choice(N, 5, replace=False)
E_col[outlier_idx] *= rng.uniform(0.5, 2.0, size=5)

df = pd.DataFrame({
    'shot_id'    : np.arange(1, N+1),
    'channel'    : ch_col,
    'energy_uJ'  : E_col.round(2),
    'duration_fs': dur_col.round(1),
})

# 2. Shots per channel
n_per_channel = df.groupby('channel')['shot_id'].count()

# 3. Per-channel stats
stats = df.groupby('channel').agg(
    mean_energy   = ('energy_uJ',   'mean'),
    stability_pct = ('energy_uJ',   lambda x: x.std()/x.mean()*100),
    mean_duration = ('duration_fs', 'mean'),
).round(2)

# 4. Outliers
df['channel_mean'] = df.groupby('channel')['energy_uJ'].transform('mean')
df['channel_std']  = df.groupby('channel')['energy_uJ'].transform('std')
df['is_outlier']   = np.abs(df['energy_uJ'] - df['channel_mean']) > 2 * df['channel_std']
n_outliers         = df['is_outlier'].sum()

# 5. Correlation
correlations = df.groupby('channel')[['energy_uJ','duration_fs']].corr()

assert n_outliers >= 1
print("Shots per channel:"); print(n_per_channel)
print("\nPer-channel statistics:"); print(stats)
print(f"\nOutliers: {n_outliers}")
print(df[df['is_outlier']][['shot_id','channel','energy_uJ']].to_string())
print("All assertions passed ✓")

---
## Exercise 6 — Save/load a pulse dataset to HDF5

In [ ]:
N_h5 = 2048; dt_h5 = 5e-15
t_h5 = (np.arange(N_h5) - N_h5//2) * dt_h5
taus = {'pulse_050fs': 50e-15, 'pulse_100fs': 100e-15, 'pulse_200fs': 200e-15}
pulses = {name: np.exp(-t_h5**2/(2*τ**2)) for name, τ in taus.items()}

with tempfile.TemporaryDirectory() as tmp:
    path_h5 = os.path.join(tmp, 'pulse_dataset.h5')

    with h5py.File(path_h5, 'w') as f:
        exp_grp = f.create_group('experiment')
        exp_grp.attrs['date']                = '2026-04-01'
        exp_grp.attrs['operator']            = 'student'
        exp_grp.attrs['laser_wavelength_nm'] = 800.0

        for name, τ_val in taus.items():
            grp = f.create_group(name)
            grp.create_dataset('t',   data=t_h5,        compression='gzip', compression_opts=6)
            grp.create_dataset('E_t', data=pulses[name], compression='gzip', compression_opts=6)
            grp['t'].attrs['units']   = 's'
            grp['E_t'].attrs['units'] = 'a.u.'
            grp.attrs['tau_fs']       = τ_val * 1e15

    file_size = os.path.getsize(path_h5)
    print(f"File size: {file_size/1024:.1f} kB")

    loaded = {}
    with h5py.File(path_h5, 'r') as f:
        f.visit(lambda name: print(f"  {name}"))
        for name in taus:
            grp = f[name]
            loaded[name] = {'t': grp['t'][:], 'E_t': grp['E_t'][:],
                            'tau_fs': grp.attrs['tau_fs']}

    for name, E_orig in pulses.items():
        ok = np.allclose(loaded[name]['E_t'], E_orig)
        print(f"{name}: round-trip {'✓' if ok else '✗'}")
        assert ok

    fig, ax = plt.subplots(figsize=(7, 3.5))
    for name, data in loaded.items():
        ax.plot(data['t']*1e15, data['E_t']**2, lw=2, label=f"τ={data['tau_fs']:.0f} fs")
    ax.set_xlabel('Time (fs)'); ax.set_ylabel('Intensity (a.u.)')
    ax.set_title('Pulses loaded from HDF5'); ax.set_xlim(-600, 600); ax.legend()
    fig.tight_layout(); plt.show()

print("All assertions passed ✓")

---
## Exercise 7 — Autocorrelation via FFT convolution

In [ ]:
N_ac = 4096; dt_ac = 5e-15; τ_ac = 100e-15; GDD_ac = 1000e-30
t_ac = (np.arange(N_ac) - N_ac//2) * dt_ac
I_TL = np.exp(-t_ac**2 / τ_ac**2)

freq_ac = np.fft.fftshift(np.fft.fftfreq(N_ac, d=dt_ac))
ω_ac    = 2 * np.pi * freq_ac
E_ac    = np.exp(-t_ac**2 / (2*τ_ac**2))
E_f_ac  = np.fft.fftshift(np.fft.fft(E_ac))
E_f_ch  = E_f_ac * np.exp(0.5j * GDD_ac * ω_ac**2)
E_chirp_ac = np.fft.ifft(np.fft.ifftshift(E_f_ch))
I_chirp = np.abs(E_chirp_ac)**2

def autocorr_fft(I):
    """Intensity autocorrelation via Wiener-Khinchin theorem."""
    I_f = np.fft.fft(I)
    return np.fft.ifft(np.abs(I_f)**2).real

A_TL    = autocorr_fft(I_TL)
A_chirp = autocorr_fft(I_chirp)

τ_axis = np.fft.fftshift(np.fft.fftfreq(N_ac, d=1/(N_ac*dt_ac)))

A_scipy = signal.correlate(I_TL, I_TL, mode='full')

Δτ_TL_meas    = fwhm(τ_axis, np.fft.fftshift(A_TL))
Δτ_TL_expect  = 2 * np.sqrt(2 * np.log(2)) * np.sqrt(2) * τ_ac
Δτ_chirp_meas = fwhm(τ_axis, np.fft.fftshift(A_chirp))

A_TL_centred    = np.fft.fftshift(A_TL) / np.fft.fftshift(A_TL).max()
A_scipy_centred = A_scipy / A_scipy.max()
mid = N_ac // 2
assert np.allclose(A_TL_centred[mid-100:mid+100],
                   A_scipy_centred[N_ac-100:N_ac+100], atol=1e-4)
assert np.isclose(Δτ_TL_meas, Δτ_TL_expect, rtol=0.02)

A_TL_plt    = np.fft.fftshift(A_TL)
A_chirp_plt = np.fft.fftshift(A_chirp)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(τ_axis*1e15, A_TL_plt/A_TL_plt.max(),
        lw=2, label=f'TL   FWHM={Δτ_TL_meas*1e15:.0f} fs')
ax.plot(τ_axis*1e15, A_chirp_plt/A_chirp_plt.max(),
        lw=2, ls='--', label=f'Chirped FWHM={Δτ_chirp_meas*1e15:.0f} fs')
ax.set_xlabel('Delay τ (fs)'); ax.set_ylabel('Autocorrelation (norm.)')
ax.set_title(f'Intensity autocorrelation  (GDD={GDD_ac*1e30:.0f} fs²)')
ax.set_xlim(-1500, 1500); ax.legend()
fig.tight_layout(); plt.show()

print(f"TL   AC FWHM : {Δτ_TL_meas*1e15:.1f} fs  (expected {Δτ_TL_expect*1e15:.1f} fs)")
print(f"Chirped AC FWHM: {Δτ_chirp_meas*1e15:.1f} fs")
print()
print("The chirped pulse has a broader autocorrelation, reflecting its longer")
print("temporal intensity envelope. The intensity autocorrelation is sensitive")
print("to the pulse duration but NOT to the spectral phase (chirp). It cannot")
print("distinguish a TL pulse from a chirped one of the same duration — that")
print("requires interferometric autocorrelation or FROG/SPIDER.")
print("\nAll assertions passed ✓")